# Checkpoint 5F — UK-DALE remaining four windows, five-seed LIME

Copy and run **the single code cell below** in the same Colab runtime that passed Checkpoint 5E. It verifies the **exact D5 UK-DALE 8,192-row training reference**, reproduces archived **seed-42 LIME for each of UKDALE_02–05**, then varies only the LIME perturbation seed over 42, 123, 456, 789, 2026. It records all 20 LIME runs, complete original-feature mappings, Jaccard comparisons and surrogate fidelity, writing only a new `CHECKPOINT5F` evidence directory after all checks pass. No Isolation Forest retraining, no SHAP, no new thresholds or historical overwrites.

In [ ]:
# CHECKPOINT 5F — UK-DALE remaining four windows: five-seed regression LIME
# Same frozen 4B protocol; only LIME perturbation seed varies.
# No detector training, threshold changes, SHAP, or historical overwrites.

from pathlib import Path
from itertools import combinations
import hashlib
import importlib.metadata as im
import json
import joblib
import numpy as np
import pandas as pd
from lime.lime_tabular import LimeTabularExplainer

required = ['ROOT', 'D5', 'FEATURES', 'COLUMNS', 'STABILITY_PROTOCOL', 'protocol_hash']
missing = [name for name in required if name not in globals()]
assert not missing, f'STOP: Missing verified prior runtime variables: {missing}'

# Gate A: locked software, protocol and 23-feature mapping.
for pkg, expected in {
    'scikit-learn': '1.6.1', 'lime': '0.2.0.1',
    'numpy': '2.4.6', 'pandas': '2.2.3'
}.items():
    assert im.version(pkg) == expected, f'STOP: {pkg} changed'

ROOT = Path(ROOT)
D5 = Path(D5)
p = STABILITY_PROTOCOL
canonical = json.dumps(p, sort_keys=True, separators=(',', ':'))
assert hashlib.sha256(canonical.encode('utf-8')).hexdigest() == protocol_hash
assert protocol_hash == '849e9d7ef15cc3b135ac74854f491ae3c2113f39c3ef527ca789cde8e829334c'
protocol_file = ROOT / 'SUPERVISOR_FINAL_EXPERIMENTS_2026' / '01_protocol' / 'CHECKPOINT4B_STABILITY_13WINDOW_849e9d7e.json'
assert protocol_file.is_file()
assert hashlib.sha256(protocol_file.read_bytes()).hexdigest() == protocol_hash

seeds = [42, 123, 456, 789, 2026]
case_ids = ['UKDALE_02', 'UKDALE_03', 'UKDALE_04', 'UKDALE_05']
assert p['seeds'] == seeds and list(FEATURES) == p['feature_order']
assert len(FEATURES) == len(COLUMNS) == 23
assert list(COLUMNS) == [f'{name}_z' for name in FEATURES]
assert p['datasets']['UK-DALE'] == 5
lp = p['lime']
assert lp['mode'] == 'regression' and lp['vary'] == 'perturbation_random_state'
assert lp['reference_size'] == 8192 and lp['reference_seeds']['UK-DALE'] == 2026
assert lp['num_samples'] == 5000 and lp['num_features'] == 10
assert lp['feature_selection'] == 'auto' and lp['discretizer'] == 'quartile'
assert lp['sample_around_instance'] is False
assert abs(float(lp['kernel_width']) - 0.75 * np.sqrt(23)) < 1e-12

# Confirm completed 5E SHAP evidence for the same four windows (read-only).
shap_summary_file = (ROOT / 'SUPERVISOR_FINAL_EXPERIMENTS_2026' /
    '06_explanation_evaluation' /
    'CHECKPOINT5E_UKDALE_REMAINING4_SHAP_5SEEDS_849e9d7e' /
    'ukdale_remaining4_shap_window_summary.csv')
assert shap_summary_file.is_file(), 'STOP: Checkpoint 5E evidence missing'
shap_summary = pd.read_csv(shap_summary_file)
assert set(shap_summary['window_id']) == set(case_ids) and len(shap_summary) == 4
assert shap_summary['protocol_sha256'].eq(protocol_hash).all()
assert (shap_summary['max_additivity_residual'] <= 1e-6).all()

manifest = json.loads((D5 / 'manifest.json').read_text(encoding='utf-8'))
entry = manifest['model_and_background']['UK-DALE']
model_path = Path(entry['model_path'])
score_path = Path(entry['scoring_file'])
assert model_path.is_file() and score_path.is_file()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

assert sha256_file(model_path) == entry['model_sha256']
model_uk5f = joblib.load(model_path)  # previously authenticated user-owned model
assert model_uk5f.n_features_in_ == 23 and len(model_uk5f.estimators_) == 300
assert model_uk5f.max_samples_ == 256 and abs(float(model_uk5f.offset_) + 0.5) <= 1e-12

def score_fn_uk5f(X):
    Z = np.asarray(X, dtype=np.float32)
    if Z.ndim == 1:
        Z = Z.reshape(1, -1)
    assert Z.ndim == 2 and Z.shape[1] == 23 and np.isfinite(Z).all()
    return -model_uk5f.decision_function(Z)

# Gate B: original D5 normalized inputs and saved explanation evidence.
selected_path = D5 / 'selected_scaled_inputs_13_windows.csv'
fidelity_path = D5 / 'lime_fidelity_13_windows.csv'
coef_path = D5 / 'regression_lime_coefficients_130_rows.csv'
for path in [selected_path, fidelity_path, coef_path]:
    assert path.is_file() and sha256_file(path) == manifest['artifact_sha256'][path.name], (
        f'STOP: D5 evidence hash mismatch for {path.name}')
selected_disk = pd.read_csv(selected_path, float_precision='round_trip')
assert len(selected_disk) == 13 and selected_disk['window_id'].is_unique
assert selected_disk['dataset'].value_counts().to_dict() == {'UK-DALE': 5, 'AMPds2': 8}
cases = selected_disk.set_index('window_id').loc[case_ids]
assert cases['dataset'].eq('UK-DALE').all()
inputs = cases[COLUMNS].to_numpy(dtype=np.float32)
assert inputs.shape == (4, 23) and np.isfinite(inputs).all()
archived_scores = cases['anomaly_score'].to_numpy(dtype=np.float64)
observed_scores = score_fn_uk5f(inputs)
assert np.max(np.abs(observed_scores - archived_scores)) <= 1e-9

old_fidelity = pd.read_csv(fidelity_path).set_index('window_id').loc[case_ids]
assert old_fidelity['dataset'].eq('UK-DALE').all()
assert old_fidelity['lime_mode'].eq('regression').all()
assert old_fidelity['lime_random_state'].eq(42).all()
assert old_fidelity['lime_num_samples'].eq(5000).all()
assert old_fidelity['lime_num_features'].eq(10).all()
assert np.max(np.abs(old_fidelity['lime_target_prediction'].to_numpy(dtype=float) - observed_scores)) <= 1e-9
old_coeff = pd.read_csv(coef_path)
assert len(old_coeff.loc[old_coeff['window_id'].isin(case_ids)]) == 40
print('PASS: Frozen protocol, verified SHAP batch, IF model and four D5 targets')

# Gate C: EXACT historical 8192-row D5 UK training reference in scoring-CSV order.
N_TRAIN = 218358
REF_N = 8192
positions = np.sort(np.random.default_rng(2026).choice(N_TRAIN, size=REF_N, replace=False))
assert positions.size == REF_N and len(np.unique(positions)) == REF_N
reference_parts = []
training_seen = 0
for chunk in pd.read_csv(score_path, usecols=['split'] + list(COLUMNS),
                         chunksize=75000, float_precision='round_trip',
                         low_memory=False):
    mask = chunk['split'].astype(str).eq('train')
    count = int(mask.sum())
    if count == 0:
        continue
    lower = np.searchsorted(positions, training_seen, side='left')
    upper = np.searchsorted(positions, training_seen + count, side='left')
    if upper > lower:
        local_positions = positions[lower:upper] - training_seen
        reference_parts.append(
            chunk.loc[mask, COLUMNS].iloc[local_positions].to_numpy(dtype=np.float32))
    training_seen += count
assert training_seen == N_TRAIN and reference_parts
reference_uk5f = np.vstack(reference_parts)
assert reference_uk5f.shape == (REF_N, 23)
assert reference_uk5f.dtype == np.float32 and np.isfinite(reference_uk5f).all()
reference_hash = hashlib.sha256(reference_uk5f.tobytes(order='C')).hexdigest()
position_hash = hashlib.sha256(positions.astype('<i8').tobytes()).hexdigest()
assert reference_hash == entry['training_reference_sha256'], 'STOP: Original D5 reference mismatched'
assert position_hash == entry['training_position_sha256'], 'STOP: D5 sampled positions mismatched'
print('PASS: 218,358 training rows and exact 8,192-row D5 LIME reference hashes')

# Gate D: run 4 windows x 5 seeds, with seed-42 historical reproduction for EACH case.
runs, feature_records, pair_records, window_summaries = [], [], [], []
for case_number, window_id in enumerate(case_ids):
    target = inputs[case_number]
    score = float(observed_scores[case_number])
    archived_row = old_fidelity.loc[window_id]
    archived_coefficients = old_coeff.loc[old_coeff['window_id'].eq(window_id)].sort_values('rank')
    assert len(archived_coefficients) == 10
    assert archived_coefficients['rank'].tolist() == list(range(1, 11))
    assert archived_coefficients['lime_mode'].eq('regression').all()
    top_sets = {}
    case_runs = []
    for seed in seeds:
        # Fixed data, model and LIME configuration; ONLY perturbation seed changes.
        explainer = LimeTabularExplainer(
            training_data=reference_uk5f, feature_names=list(COLUMNS),
            mode='regression', categorical_features=[],
            discretize_continuous=True, discretizer='quartile',
            feature_selection='auto', sample_around_instance=False,
            kernel_width=0.75 * np.sqrt(23), random_state=seed)
        explanation = explainer.explain_instance(
            data_row=target, predict_fn=score_fn_uk5f,
            num_features=10, num_samples=5000)
        r2 = float(explanation.score)
        local_pred = float(np.asarray(explanation.local_pred).reshape(-1)[0])
        predicted = float(np.asarray(explanation.predicted_value).reshape(-1)[0])
        assert np.isfinite([r2, local_pred, predicted]).all() and r2 <= 1.0 + 1e-8
        assert abs(predicted - score) <= 1e-9
        mapping = explanation.as_map()[1]
        weights = {int(index): float(weight) for index, weight in mapping}
        assert len(mapping) == len(weights) == 10
        assert all(0 <= index < 23 and np.isfinite(weight) for index, weight in weights.items())
        order = sorted(weights, key=lambda index: (-abs(weights[index]), index))
        assert len(order) == 10
        assert abs(weights[order[4]]) > 1e-12, 'STOP: Unselected/zero feature in top five'
        top_five = order[:5]
        top_sets[seed] = set(top_five)

        if seed == 42:
            assert abs(r2 - float(archived_row['lime_local_r2'])) <= 1e-6, (
                f'STOP: {window_id} seed42 R2 differs from D5')
            assert abs(local_pred - float(archived_row['lime_surrogate_prediction'])) <= 1e-6, (
                f'STOP: {window_id} seed42 local prediction differs from D5')
            expected_names = archived_coefficients['feature'].tolist()
            assert [FEATURES[index] for index in order] == expected_names, (
                f'STOP: {window_id} seed42 10-feature mapping/ranking differs from D5')
            expected_weights = archived_coefficients['lime_weight'].to_numpy(dtype=float)
            observed_weights = np.asarray([weights[index] for index in order])
            assert np.max(np.abs(observed_weights - expected_weights)) <= 1e-6, (
                f'STOP: {window_id} seed42 weights differ from D5')
            print(f'PASS: {window_id} seed-42 archived D5 LIME reproduced')

        run = dict(dataset='UK-DALE', window_id=window_id, seed=seed,
                   anomaly_score=score, local_r2=r2,
                   surrogate_prediction=local_pred,
                   absolute_local_prediction_error=abs(local_pred - score),
                   selected_feature_count=10,
                   top5_features='|'.join(FEATURES[index] for index in top_five))
        runs.append(run)
        case_runs.append(run)
        for index, feature in enumerate(FEATURES):
            feature_records.append(dict(
                dataset='UK-DALE', window_id=window_id, seed=seed,
                feature_index=index, feature=feature,
                scaled_feature_value=float(target[index]),
                selected_in_surrogate=(index in weights),
                lime_weight=weights.get(index, None),  # missing is NOT a zero coefficient
                top5=(index in top_sets[seed])))
        print(f'PASS {window_id} seed={seed}: R2={r2:.6f}; '
              f'local_error={abs(local_pred-score):.6f}; '
              f'top5={[FEATURES[index] for index in top_five]}')

    case_pairs = []
    for seed_a, seed_b in combinations(seeds, 2):
        A, B = top_sets[seed_a], top_sets[seed_b]
        row = dict(dataset='UK-DALE', window_id=window_id,
                   seed_a=seed_a, seed_b=seed_b,
                   intersection_count=len(A & B), union_count=len(A | B),
                   top5_jaccard=len(A & B) / len(A | B))
        pair_records.append(row)
        case_pairs.append(row)
    assert len(case_pairs) == 10
    jac = np.asarray([v['top5_jaccard'] for v in case_pairs], dtype=np.float64)
    r2_values = np.asarray([v['local_r2'] for v in case_runs], dtype=np.float64)
    window_summaries.append(dict(
        dataset='UK-DALE', window_id=window_id, protocol_sha256=protocol_hash,
        method='LIME', vary='perturbation_random_state', reference_seed=2026,
        reference_sha256=reference_hash, n_runs=5, n_seed_pairs=10,
        mean_pairwise_top5_jaccard=float(jac.mean()),
        min_pairwise_top5_jaccard=float(jac.min()),
        max_pairwise_top5_jaccard=float(jac.max()),
        mean_local_r2=float(r2_values.mean()),
        sample_sd_local_r2=float(r2_values.std(ddof=1)),
        min_local_r2=float(r2_values.min()),
        max_local_r2=float(r2_values.max()),
        mean_absolute_local_error=float(np.mean([
            r['absolute_local_prediction_error'] for r in case_runs]))))

assert len(runs) == 20 and len(feature_records) == 460
assert len(pair_records) == 40 and len(window_summaries) == 4
assert len({(r['window_id'], r['seed']) for r in runs}) == 20
assert len({(r['window_id'], r['seed'], r['feature_index']) for r in feature_records}) == 460

# Gate E: save NEW evidence only after all 20 explanations and D5 gates pass.
parent = ROOT / 'SUPERVISOR_FINAL_EXPERIMENTS_2026' / '06_explanation_evaluation'
assert parent.is_dir()
output = parent / 'CHECKPOINT5F_UKDALE_REMAINING4_LIME_PERTURBATION_5SEEDS_849e9d7e'

def as_csv_bytes(records):
    return pd.DataFrame(records).to_csv(
        index=False, float_format='%.17g').encode('utf-8')

batch_info = dict(dataset='UK-DALE', window_ids=case_ids,
                  protocol_sha256=protocol_hash, method='LIME',
                  treatment='perturbation_seed_only', fixed_reference_size=8192,
                  reference_seed=2026, reference_sha256=reference_hash,
                  seeds=seeds, n_windows=4, n_runs=20,
                  n_pairwise_comparisons=40, n_feature_records=460,
                  archived_seed42_numerically_reproduced=True,
                  historical_files_unchanged=True)
files_to_save = {
    'ukdale_remaining4_lime_runs.csv': as_csv_bytes(runs),
    'ukdale_remaining4_lime_23_feature_mapping.csv': as_csv_bytes(feature_records),
    'ukdale_remaining4_lime_pairwise_jaccard.csv': as_csv_bytes(pair_records),
    'ukdale_remaining4_lime_window_summary.csv': as_csv_bytes(window_summaries),
    'ukdale_remaining4_lime_batch_manifest.json': (
        json.dumps(batch_info, indent=2, sort_keys=True, allow_nan=False) + '\n'
    ).encode('utf-8')
}
for filename, contents in files_to_save.items():
    path = output / filename
    if path.exists():
        assert path.read_bytes() == contents, f'STOP: Existing new evidence differs: {path}'
output.mkdir(exist_ok=True)
for filename, contents in files_to_save.items():
    path = output / filename
    if not path.exists():
        with path.open('xb') as stream:
            stream.write(contents)
    assert path.read_bytes() == contents

print('\n=== CHECKPOINT 5F SUMMARY ===')
print(pd.DataFrame(window_summaries)[[
    'window_id', 'mean_pairwise_top5_jaccard',
    'min_pairwise_top5_jaccard', 'mean_local_r2',
    'sample_sd_local_r2', 'mean_absolute_local_error'
]].to_string(index=False))
print('Total verified LIME explanations:', len(runs))
print('Total original-feature mapping records:', len(feature_records))
print('Evidence directory:', output)
print('\nPASS: UKDALE_REMAINING4_LIME_FIVE_SEED_EXPERIMENT')
print('No detector retraining, SHAP, threshold updates, or historical overwrites.')
